# Seguro médico: estimar el coste anual

Este cuaderno recorre el proyecto de regresión: cargar y revisar datos, limpiar, explorar relaciones, comparar modelos y probar una estimación. La variable objetivo es `charges`, el coste anual en dólares.

**En sencillo:** usamos edad, IMC, número de hijos, sexo, consumo de tabaco y región para estimar el coste anual observado en ejemplos anteriores. Es un ejercicio didáctico, no una oferta de seguro ni asesoramiento médico.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ml_al_alfa.data import clean_insurance_data, feature_defaults
from ml_al_alfa.datasets import load_insurance_frame
from ml_al_alfa.train import ARTIFACTS_DIR, RAW_DIR, train_and_save

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd()
print(f"Carpeta del proyecto: {PROJECT_ROOT}")

## 1. Cargar y revisar los datos

Si el CSV todavía no existe, la función lo descarga de la fuente pública y lo guarda en `data/raw/`.

In [ ]:
raw = load_insurance_frame(RAW_DIR)
print(f"Filas y columnas: {raw.shape}")
display(raw.head())
display(raw.isna().sum().rename("valores ausentes").to_frame())
print(f"Filas duplicadas: {raw.duplicated().sum()}")

## 2. Limpiar

La limpieza elimina duplicados y filas con valores imposibles. `sex` y `smoker` se convierten a 0/1; las categorías de región se mantienen para que el pipeline las codifique. Los campos que no forman parte de la pregunta se descartan.

In [ ]:
X, y, cleaning_report = clean_insurance_data(raw)
display(pd.Series(cleaning_report.as_dict(), name="recuento").to_frame())
display(X.head())
print(f"Objetivo: coste anual en USD; filas limpias: {len(X):,}.")

## 3. Explorar visualmente

Los gráficos ayudan a ver la distribución de los costes y algunas relaciones. Una relación visible no demuestra por sí sola que una característica cause un coste mayor.

In [ ]:
plot_data = X.copy()
plot_data["charges"] = y
plot_data["smoker_label"] = plot_data["smoker"].map({0: "No fuma", 1: "Fuma"})
plot_data["age_group"] = pd.cut(
    plot_data["age"], bins=[17, 25, 35, 45, 55, 65, 101],
    labels=["18–25", "26–35", "36–45", "46–55", "56–65", "66+"],
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(plot_data["charges"], bins=35, ax=axes[0])
axes[0].set(title="Distribución del coste anual", xlabel="Coste (USD)")
sns.scatterplot(data=plot_data, x="bmi", y="charges", hue="smoker_label", alpha=0.65, ax=axes[1])
axes[1].set(title="IMC y coste según tabaquismo", xlabel="Índice de masa corporal", ylabel="Coste (USD)")
sns.barplot(data=plot_data, x="age_group", y="charges", errorbar=None, ax=axes[2])
axes[2].tick_params(axis="x", rotation=30)
axes[2].set(title="Coste medio por edad", xlabel="Edad", ylabel="Coste medio (USD)")
plt.tight_layout()
plt.show()

## 4. Entrenar y evaluar

Usamos el mismo entrenamiento que la aplicación para que el artefacto y las métricas sean coherentes con Streamlit. Separamos el test antes de ajustar los modelos y comparamos la regresión lineal y el bosque aleatorio con el baseline, que predice la media del entrenamiento.

In [ ]:
bundle = train_and_save(
    X,
    y,
    dataset_name="insurance",
    rows=cleaning_report.as_dict(),
    output_path=ARTIFACTS_DIR / "insurance_model.joblib",
)
metrics = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(metrics.rename(columns={"mae": "MAE (USD, menor es mejor)", "r2": "R²"}))
print(f"Mejor modelo: {bundle['model_name']}")
print(f"Modelo para Streamlit: {ARTIFACTS_DIR / 'insurance_model.joblib'}")

## 5. Probar una estimación

Este ejemplo usa valores típicos de los datos limpios. Cambia los valores para probar otros perfiles; el resultado es una estimación del modelo, no una cotización.

In [ ]:
sample = pd.DataFrame([feature_defaults(X)], columns=bundle["features"])
sample_prediction = float(bundle["pipeline"].predict(sample)[0])
display(sample)
print(f"Coste anual estimado: ${sample_prediction:,.2f} USD")
print(f"MAE de prueba: {bundle['metrics'][bundle['model_name']]['mae']:,.2f} USD")